In [1]:
import pandas as pd

# load data 
data = pd.read_csv('../data/processed/danish_croissant_data.csv')

data['date'] = pd.to_datetime(data['date'])

# Create full date range 
date_range = pd.date_range(start= data['date'].min(),
                            end = data['date'].max(),
                            freq='D')

print(f"Expected days: {len(date_range)}")
print(f"Actual days in data: {data['date'].nunique()}")
print(f"Missing days: {len(date_range) - data['date'].nunique()}")

Expected days: 1632
Actual days in data: 972
Missing days: 660


# Regime Change

In [2]:
data['day_of_week'] = data['date'].dt.dayofweek
data['day_name'] = data['date'].dt.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Count salex by day of week for each year 
yearly_patterns = data.groupby([data['date'].dt.year, 'day_name']).size().unstack(fill_value=0)
yearly_patterns = yearly_patterns.reindex(columns=day_order, fill_value= 0)
print(yearly_patterns)

day_name  Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
date                                                                    
2021           1       40         66        66      66        64      62
2022           0        1         56        90     101       101     100
2023           0        4         28        68      96        97      97
2024          36        2          0        71      92       101      98
2025           0        2          5        59      74        81      77


In [3]:
from doughflow.data.make_dataset import filter_operational_days

operating_data = filter_operational_days(data)

operating_data

,date,item,quantity,day_name
0,2021-05-13,Croissant,42,Thursday
1,2021-05-13,Danish,27,Thursday
2,2021-05-14,Croissant,61,Friday
3,2021-05-14,Danish,27,Friday
4,2021-05-15,Croissant,100,Saturday
...,...,...,...,...
1656,2025-10-25,Danish,42,Saturday
1657,2025-10-26,Croissant,37,Sunday
1658,2025-10-26,Danish,13,Sunday
1659,2025-10-30,Croissant,17,Thursday


## Find Missing Values in Operational Days

In [4]:
# operational days
operational_dates = pd.date_range(
    start= operating_data['date'].min(),
    end = operating_data['date'].max(),
    freq = 'D'
)

# Only Thu-Sun
operational_dates = operational_dates[operational_dates.dayofweek.isin([3,4,5,6])]

# Find missing operational days
actual_dates = set(operating_data['date'])
expected_dates = set(operational_dates)
missing_operational_days = expected_dates - actual_dates

print(f"Actual days: {len(actual_dates)}")
print(f"expected_dates: {len(expected_dates)}")
print(f"Missing operational days: {len(missing_operational_days)}")
for date in sorted(missing_operational_days)[:5]:
    print(f"  {date.strftime('%Y-%m-%d %A')}")

Actual days: 846
expected_dates: 933
Missing operational days: 87
  2021-07-04 Sunday
  2021-11-25 Thursday
  2021-11-26 Friday
  2021-12-25 Saturday
  2021-12-26 Sunday


## Check Holidays

In [5]:
import holidays

us_holidays = holidays.US()

for date in sorted(missing_operational_days):
    if date in us_holidays:
        print(f"{date}: {us_holidays[date]} - Bakery likely closed")
    else: 
        print(f"{date}: UNEXPECTED MISSING - investigate!")


2021-07-04 00:00:00: Independence Day - Bakery likely closed
2021-11-25 00:00:00: Thanksgiving Day - Bakery likely closed
2021-11-26 00:00:00: UNEXPECTED MISSING - investigate!
2021-12-25 00:00:00: Christmas Day - Bakery likely closed
2021-12-26 00:00:00: UNEXPECTED MISSING - investigate!
2022-01-01 00:00:00: New Year's Day - Bakery likely closed
2022-04-14 00:00:00: UNEXPECTED MISSING - investigate!
2022-04-17 00:00:00: UNEXPECTED MISSING - investigate!
2022-08-25 00:00:00: UNEXPECTED MISSING - investigate!
2022-09-01 00:00:00: UNEXPECTED MISSING - investigate!
2022-09-29 00:00:00: UNEXPECTED MISSING - investigate!
2022-11-24 00:00:00: Thanksgiving Day - Bakery likely closed
2022-11-25 00:00:00: UNEXPECTED MISSING - investigate!
2022-11-26 00:00:00: UNEXPECTED MISSING - investigate!
2022-12-22 00:00:00: UNEXPECTED MISSING - investigate!
2022-12-25 00:00:00: Christmas Day - Bakery likely closed
2023-01-01 00:00:00: New Year's Day - Bakery likely closed
2023-01-05 00:00:00: UNEXPECTED M

# Testing complete pipeline

In [3]:
# Load & filter operational days
from doughflow.data.make_dataset import filter_operational_days, create_complete_date_range, fill_holidays_missing_values, rename_data
from doughflow.features.build_features import TemporalFeatureExtractor, LagFeatureExtractor
import holidays 
import pandas as pd

df = pd.read_csv('../data/processed/danish_croissant_data.csv')


operating_data = filter_operational_days(df)

# Create complete date range 
complete_data = create_complete_date_range(operating_data)

# Fill missing values
us_holidays = holidays.US()
data_with_filled_holidays = fill_holidays_missing_values(complete_data, us_holidays)

# Apply feature engineering to handle remaining missing values
temporal_extractor = TemporalFeatureExtractor()
data_with_temporal = temporal_extractor.fit_transform(data_with_filled_holidays)

# Extract lag features 
lag_extractor = LagFeatureExtractor(fill_strategy='cascading')
final_data = lag_extractor.fit_transform(data_with_temporal)

print(f"Final NaN count: {final_data.isna().sum().sum()}")


Final NaN count: 355
